# 06: Temporal Analysis

**What this notebook does:** `04` and `05` each treated every locality-year as an independent
data point. This notebook does the thing that was deliberately left undone in both -- follows
the *same* locality across its available election years (2010 through 2025, three-year cycles)
and asks whether its cluster membership from `04` (landslide-type vs. competitive-type local
races) stays put or shifts.

Not every locality has all six years on record -- `01`'s coverage audit and `04`'s complete-case
filtering both drop rows, and they don't always drop the same ones for the same locality. That's
handled directly below rather than assumed away: transitions are only computed between
*consecutive observed* elections for a given locality, and localities with fewer than two
observed years are reported separately rather than silently excluded.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("..") / "src"))

%matplotlib inline
pd.set_option("display.max_columns", 40)
print("pandas:", pd.__version__)

pandas: 3.0.2


In [2]:
CONFIG = {
    "processed_dir": "../data/processed",
}

processed_dir = Path(CONFIG["processed_dir"])
print("Config set.")

Config set.


## Load `04`'s cluster assignments

In [3]:
locality_clusters = pd.read_parquet(processed_dir / "locality_clusters.parquet")
print("Locality clusters:", locality_clusters.shape)
print("Years:", sorted(locality_clusters["year"].unique()))
print("Cluster labels present:", sorted(locality_clusters["cluster"].unique()))

Locality clusters: (8967, 14)
Years: [np.int64(2010), np.int64(2013), np.int64(2016), np.int64(2019), np.int64(2022), np.int64(2025)]
Cluster labels present: [np.int32(0), np.int32(1)]


## How much history does each locality actually have?

Confirmed by counting rather than assumed: most localities do have all six years, but a
meaningful minority don't, and a "transition" needs at least two.

In [4]:
years_per_locality = locality_clusters.groupby(["province", "city"])["year"].apply(sorted)
n_years = years_per_locality.apply(len)

print("Distribution of number of observed election years per locality:")
display(n_years.value_counts().sort_index().to_frame("n_localities"))

single_year = (n_years == 1).sum()
print(f"\n{single_year} localities have only one observed year -- excluded from transition "
      f"analysis below (nothing to transition from/to), kept in the raw history table.")

Distribution of number of observed election years per locality:


,n_localities
year,
1,60
2,68
3,105
4,96
5,634
6,817



60 localities have only one observed year -- excluded from transition analysis below (nothing to transition from/to), kept in the raw history table.


## Build the per-locality cluster history and consecutive transitions

In [5]:
history = (locality_clusters.pivot_table(index=["province", "city"], columns="year",
                                            values="cluster", aggfunc="first"))
print("Cluster-history matrix (one row per locality, one column per election year):")
display(history.head())

transitions = []
for (province, city), years in years_per_locality.items():
    if len(years) < 2:
        continue
    for y_from, y_to in zip(years[:-1], years[1:]):
        c_from = history.loc[(province, city), y_from]
        c_to = history.loc[(province, city), y_to]
        transitions.append({"province": province, "city": city, "year_from": y_from,
                             "year_to": y_to, "cluster_from": c_from, "cluster_to": c_to,
                             "changed": c_from != c_to})

transitions = pd.DataFrame(transitions)
print(f"\n{len(transitions):,} consecutive-election transitions across "
      f"{transitions[['province', 'city']].drop_duplicates().shape[0]:,} localities.")

Cluster-history matrix (one row per locality, one column per election year):


year                2010  2013  2016  2019  2022  2025
province city                                         
ABRA     BANGUED     1.0   1.0   1.0   0.0   1.0   0.0
         BOLINEY     1.0   NaN   0.0   0.0   0.0   0.0
         BUCAY       1.0   0.0   0.0   0.0   0.0   0.0
         BUCLOC      1.0   NaN   1.0   0.0   1.0   1.0
         DAGUIOMAN   1.0   NaN   0.0   0.0   0.0   0.0


7,187 consecutive-election transitions across 1,720 localities.


## Transition matrix

Since `04` settled on k=2 (a landslide-type cluster and a competitive-type cluster), this is a
simple 2x2: given a locality was in cluster X at one election, what fraction were in cluster Y
at the next.

In [6]:
trans_matrix = pd.crosstab(transitions["cluster_from"], transitions["cluster_to"], normalize="index")
print("P(cluster at next election | cluster at this election):")
display(trans_matrix.round(3))

overall_change_rate = transitions["changed"].mean()
print(f"\nOverall: {overall_change_rate:.1%} of consecutive elections saw a locality change "
      f"cluster; {1 - overall_change_rate:.1%} stayed in the same cluster.")

stay_0 = trans_matrix.loc[0.0, 0.0]
stay_1 = trans_matrix.loc[1.0, 1.0]
more_sticky, less_sticky = (1.0, 0.0) if stay_1 > stay_0 else (0.0, 1.0)
print(f"\nNotably asymmetric: cluster {int(more_sticky)} persists {trans_matrix.loc[more_sticky, more_sticky]:.1%} "
      f"of the time, cluster {int(less_sticky)} only {trans_matrix.loc[less_sticky, less_sticky]:.1%} -- "
      f"one of these two vote-shape patterns is a substantially more stable locality trait than the other.")

P(cluster at next election | cluster at this election):


cluster_to,0.0,1.0
cluster_from,,
0.0,0.541,0.459
1.0,0.219,0.781



Overall: 28.3% of consecutive elections saw a locality change cluster; 71.7% stayed in the same cluster.

Notably asymmetric: cluster 1 persists 78.1% of the time, cluster 0 only 54.1% -- one of these two vote-shape patterns is a substantially more stable locality trait than the other.


## Stability: which localities never change, and which change every time?

A locality with only two observed elections that switched once looks identical, by change rate
alone, to one with six elections that switched three times -- so this looks at both the rate and
the raw count of observed elections together, not the rate in isolation.

In [7]:
locality_summary = (transitions.groupby(["province", "city"])
                     .agg(n_transitions=("changed", "size"), n_changes=("changed", "sum")))
locality_summary["change_rate"] = locality_summary["n_changes"] / locality_summary["n_transitions"]
locality_summary = locality_summary.join(n_years.rename("n_years_observed"))

always_stable = locality_summary[(locality_summary["n_changes"] == 0) & (locality_summary["n_years_observed"] >= 5)]
print(f"{len(always_stable)} localities with 5+ observed elections never changed cluster. Examples:")
display(always_stable.sample(min(5, len(always_stable)), random_state=42))

most_volatile = locality_summary[locality_summary["n_years_observed"] >= 5].sort_values(
    "n_changes", ascending=False).head(5)
print("\nMost volatile localities (5+ observed elections, most cluster changes):")
display(most_volatile)

507 localities with 5+ observed elections never changed cluster. Examples:


,,n_transitions,n_changes,change_rate,n_years_observed
province,city,,,,
EASTERN SAMAR,MASLOG,4,0,0.0,5
MISAMIS ORIENTAL,ALUBIJID,5,0,0.0,6
ZAMBOANGA DEL NORTE,SALUG,4,0,0.0,5
BULACAN,BULACAN,5,0,0.0,6
SURIGAO DEL NORTE,SAN ISIDRO,5,0,0.0,6



Most volatile localities (5+ observed elections, most cluster changes):


,,n_transitions,n_changes,change_rate,n_years_observed
province,city,,,,
EASTERN SAMAR,SULAT,5,5,1.0,6
ILOCOS NORTE,VINTAR,5,5,1.0,6
MASBATE,SAN PASCUAL,5,5,1.0,6
CAPIZ,MA AYON,5,5,1.0,6
QUIRINO,AGLIPAY,5,5,1.0,6


## Spot check: does a flagged change actually correspond to a real shift in the numbers?

Picking the single most volatile locality above and looking at its actual `MAYOR_margin` across
every observed year -- confirming a "cluster change" here means the underlying vote-shape number
genuinely moved, not that it's flickering right at a boundary the model drew somewhat
arbitrarily.

In [8]:
top_volatile_province, top_volatile_city = most_volatile.index[0]
spot_check = (locality_clusters[(locality_clusters["province"] == top_volatile_province)
                                 & (locality_clusters["city"] == top_volatile_city)]
              .sort_values("year")[["year", "cluster", "MAYOR_margin", "MAYOR_enc"]])
print(f"{top_volatile_city}, {top_volatile_province} across every observed election:")
display(spot_check)

SULAT, EASTERN SAMAR across every observed election:


,year,cluster,MAYOR_margin,MAYOR_enc
395,2010,1,0.287948,1.846869
1467,2013,0,0.657068,1.396903
3021,2016,1,0.355889,1.944352
4654,2019,0,0.635881,1.424151
6281,2022,1,0.244210,1.905884
7915,2025,0,0.763941,1.262940


## Stability by region

Not a claim about *why* -- just whether some regions show more locality-level political
volatility than others in this data, worth a closer look later.

In [9]:
region_lookup = locality_clusters.drop_duplicates(subset=["province", "city"]).set_index(["province", "city"])["region"]
locality_summary["region"] = locality_summary.index.map(region_lookup)

by_region = (locality_summary[locality_summary["n_years_observed"] >= 4]
             .groupby("region")["change_rate"].agg(["mean", "size"])
             .rename(columns={"mean": "avg_change_rate", "size": "n_localities"})
             .sort_values("avg_change_rate"))
display(by_region.round(3))

,avg_change_rate,n_localities
region,,
REGION III,0.214,126
REGION V,0.235,107
REGION IV A,0.245,139
BARMM,0.259,97
REGION IV B,0.264,69
REGION XII,0.278,46
REGION VII,0.289,125
REGION II,0.292,90
REGION IX,0.293,66


## Save

In [10]:
history_out = history.reset_index()
history_out.columns = [str(c) for c in history_out.columns]
history_out.to_parquet(processed_dir / "locality_cluster_history.parquet", index=False)

transitions.to_parquet(processed_dir / "locality_cluster_transitions.parquet", index=False)

print("Saved locality_cluster_history.parquet:", history_out.shape)
print("Saved locality_cluster_transitions.parquet:", transitions.shape)

Saved locality_cluster_history.parquet: (1780, 8)
Saved locality_cluster_transitions.parquet: (7187, 7)


## Summary

Most localities keep the same cluster membership across elections -- the transition matrix and
per-locality change rates above quantify exactly how much, rather than assuming stability or
volatility either way. A locality flagged as volatile was spot-checked against its actual
`MAYOR_margin` history rather than trusted on the cluster label alone.

**Worth remembering:** a "cluster change" reflects a shift in *vote-shape* (competitive vs.
landslide), not a claim about which party or candidate is winning -- that information isn't in
this feature set at all (deliberately, per `02`/`03`/`04`'s design).

**Deliberately not done here:** the richer 2016+ clustering variant using province/national
columns (`07`, if wanted), joining the `05` anomaly flags into this same timeline (does an
anomalous locality tend to be a volatile one?), and any geographic/map visualization of the
transition patterns.